# Fully Bayesian SAAS GP

SAAS は、入力次元が比較的高く、そのうち実際に効いている次元が少数だと考えられる場合に有効です。この Notebook では single-task 版と multi-task 版の両方を扱います。Fully Bayesian SAAS は `make_mll()` ではなく NUTS で学習します。

> この例ではドキュメントとして実行しやすいように MCMC の設定を小さくしています。実解析では `warmup_steps` と `num_samples` を増やしてください。

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf
from robotorchan.models import SaasFullyBayesianSingleTaskGP, SaasFullyBayesianMultiTaskGP

torch.manual_seed(0)
dtype = torch.double

### SAASが想定する「高次元・疎な重要変数」

入力が12次元あっても、目的関数に効く変数が数個だけなら、すべての次元に同じ自由度を与えるGPはデータ効率が悪くなる場合があります。SAAS（Sparse Axis-Aligned Subspace）は**多くの次元は影響が小さい**という事前の仮定を、ARDカーネルの逆長さ尺度に対する疎な階層事前分布として表現します。

逆長さ尺度が小さい次元では、入力を変えても関数値が大きく変わりにくいと解釈できます。これは変数を完全に削除する仕組みではなく、関連度の不確実性も含めて推論する方法です。

## 1. 高次元の合成データ

In [ ]:
n, d = 28, 12
train_X = torch.rand(n, d, dtype=dtype)
def f(X):
    return (torch.sin(2 * torch.pi * X[..., 0]) + 0.8 * (X[..., 2] - 0.5) ** 2 - 0.6 * X[..., 5]).unsqueeze(-1)
train_Y = f(train_X) + 0.03 * torch.randn(n, 1, dtype=dtype)
train_X.shape, train_Y.shape

### この例の重要次元は既知

合成関数は12変数のうち `x0`, `x2`, `x5` の3つだけに依存します。実務ではどの次元が重要かは未知ですが、この例では予測や候補提案を理解するために真の構造を把握できます。

`train_X` は `(28, 12)`、`train_Y` は `(28, 1)` です。入力次元に対して観測が少ない設定で、SAASの疎性仮定が役立つ可能性を示します。

## 2. Single-task SAAS

まず単一タスクの SAAS GP を構築します。`raw_train_X` / `raw_train_Y` はコンストラクタに渡した元データを保持します。`supports_mll=False` は仕様であり、学習には NUTS を使用します。

In [ ]:
model = SaasFullyBayesianSingleTaskGP(train_X, train_Y)
print(model.raw_train_X.shape, model.raw_train_Y.shape)
print('supports_mll =', model.supports_mll)

### Fully Bayesian学習とNUTS

通常のExact GPでは周辺対数尤度（MLL）を最大化してハイパーパラメータの点推定を得ます。Fully Bayesian SAASは、長さ尺度などのハイパーパラメータの**事後分布をMCMCで推論**します。

`fit(warmup_steps=32, num_samples=16, thinning=2)` はNUTSによる短い学習例です。warmupはサンプラーの調整、`num_samples`はサンプリング数、`thinning`は保存間隔に関わります。ここでの設定はデモ用で、収束診断を満たす保証はありません。`supports_mll=False` は意図したAPIです。

In [ ]:
model.fit(
    warmup_steps=32,
    num_samples=16,
    thinning=2,
    disable_progbar=True,
)

### 予測のスライスを読む

12次元すべてを一度に描くことは難しいため、`x0` だけを0から1まで動かし、他の11次元を0.5に固定した**1次元スライス**を可視化します。これは12次元空間全体の予測精度を示す図ではありません。

Fully BayesianモデルではMCMCサンプル間のハイパーパラメータ不確実性も考慮する必要があります。後続の図は既存コードの簡易表示です。モデルが返す事後分布のshapeとサンプル軸を確認し、平均・分散をどの軸で集約するかに注意してください。

In [ ]:
x0 = torch.linspace(0, 1, 120, dtype=dtype)
X_test = torch.full((120, d), 0.5, dtype=dtype)
X_test[:, 0] = x0
with torch.no_grad():
    posterior = model.posterior(X_test)
mean = posterior.mean.mean(dim=0).squeeze(-1)
std = posterior.variance.mean(dim=0).sqrt().squeeze(-1)
plt.figure(figsize=(7, 4))
plt.plot(x0, mean)
plt.fill_between(x0, mean - 2 * std, mean + 2 * std, alpha=0.2)
plt.xlabel('x0')
plt.ylabel('posterior')
plt.title('SAAS posterior slice')
plt.show()

### 追加実装例：既知の重要次元を可視化

SAASの狙いを理解するため、真の合成関数について、重要な3次元と無関係な1次元をそれぞれ動かして比較します。**これは真の関数の感度を示す教育用の図**であり、学習済みSAASモデルが変数重要度を正しく推定した証拠ではありません。

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(13, 3), sharey=True)
for ax, dim in zip(axes, (0, 2, 5, 7)):
    grid = torch.linspace(0.0, 1.0, 100, dtype=dtype)
    slice_X = torch.full((100, d), 0.5, dtype=dtype)
    slice_X[:, dim] = grid
    ax.plot(grid, f(slice_X).squeeze(-1))
    ax.set_title(f"true function: x{dim}")
    ax.set_xlabel(f"x{dim}")
axes[0].set_ylabel("objective")
plt.tight_layout()
plt.show()

## 3. Multi-task SAAS

次に BoTorch の long-format task feature を使う multi-task SAAS を構築します。最後の列を `task_feature` として扱います。

In [ ]:
x_mt = torch.rand(18, d, dtype=dtype)
X0 = torch.cat([x_mt, torch.zeros(18, 1, dtype=dtype)], dim=-1)
X1 = torch.cat([x_mt, torch.ones(18, 1, dtype=dtype)], dim=-1)
Y0 = f(x_mt)
Y1 = 0.7 * f(x_mt) + 0.2
train_X_mt = torch.cat([X0, X1], dim=0)
train_Y_mt = torch.cat([Y0, Y1], dim=0)
mt_model = SaasFullyBayesianMultiTaskGP(train_X_mt, train_Y_mt, task_feature=-1, output_tasks=[1])
print(mt_model.raw_train_X.shape, mt_model.raw_train_Y.shape)
print('supports_mll =', mt_model.supports_mll)

In [ ]:
mt_model.fit(
    warmup_steps=24,
    num_samples=12,
    thinning=2,
    disable_progbar=True,
)

### Multi-task SAASのtask列

`train_X_mt` は設計変数12列とtask IDの1列から構成されます。`output_tasks=[1]` は**task 1の予測・候補提案を対象にする**指定です。task IDは最適化する連続設計変数ではありません。

ここではtask 0の応答 `f(x)` とtask 1の応答 `0.7*f(x)+0.2` を関連データとして扱います。両taskの相関を活用できるかどうかはモデル仮定と観測データに依存します。

## 4. 使い所と注意点

- 高次元で、重要な入力次元が少数だと考えられる場合に SAAS を検討します。
- 計算時間を重視する場合は通常 GP や MAP-SAAS 系も候補です。
- `supports_mll=False` は意図した仕様で、学習経路は NUTS です。
- MCMC は比較的重いため、高速な PR 用 CI では実行を省略するか、さらに軽い設定を使用してください。

### SAASの学習コストとモデル選択

Fully Bayesian SAASはMCMCを使うため、学習時間や収束診断が重要です。入力が高次元でも、重要変数が多数ある場合やMCMCのコストが許容できない場合は、通常のGP、MAP-SAAS、次元削減なども候補になります。

短いMCMC設定で得た予測や候補をそのまま本番運用の品質とみなさず、チェーンの収束・感度・再現性を確認してください。

## SAAS single-task candidate generation

NUTSで学習したfully Bayesian posteriorを使ってqLogEIを最適化します。MultiTaskの学習結果は別モデルであり、ここではsingle-taskモデルの候補を評価します。

### 獲得関数から候補評価まで

`qLogExpectedImprovement` は現在の観測最大値 `best_f` を基準に、改善が期待される設計点を選びます。`bounds` は12次元の全設計変数に対して `[0,1]` を指定し、`q=1` は候補1点を意味します。

**SAASはサロゲートの種類**であり、最適化ポリシーそのものではありません。通常のBoTorch獲得関数と候補最適化を組み合わせられる点が重要です。

In [ ]:
bounds = torch.stack([torch.zeros(d, dtype=dtype), torch.ones(d, dtype=dtype)])
acquisition = qLogExpectedImprovement(model=model, best_f=train_Y.max())
candidate, acquisition_value = optimize_acqf(
    acq_function=acquisition,
    bounds=bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
)
candidate_Y = f(candidate)
assert candidate.shape == (1, d)
assert torch.all((candidate >= bounds[0]) & (candidate <= bounds[1]))
print("candidate:", candidate)
print("acquisition value:", acquisition_value)
print("evaluated objective:", candidate_Y)


### Mixed SAASとMulti-task SAASの違い

Multi-taskではtask IDがデータ源を識別し、候補生成時にtask 1へ固定します。Mixed SAASではカテゴリが**選択可能な設計変数**です。`cat_dims` でカテゴリ列を指定し、`optimize_acqf_mixed` で有効なカテゴリ値を列挙して探索します。

MCMCは短いデモ設定のため、候補の品質を保証しません。

## 5. MultiTask・Mixed SAASの候補提案

NUTS学習後のモデルを使い、BoTorchのqLogEIで候補を提案します。MultiTaskは`output_tasks=[1]`で対象タスクを固定し、Mixed型はカテゴリを離散列挙します。MCMCの短い設定は動作確認用であり、収束・予測精度を保証しません。


In [ ]:
from botorch.optim.optimize import optimize_acqf_mixed

from robotorchan.models import (
    MixedSaasFullyBayesianMultiTaskGP,
    MixedSaasFullyBayesianSingleTaskGP,
)

# MultiTask SAAS: optimize design inputs for the selected output task.
mt_acquisition = qLogExpectedImprovement(
    model=mt_model,
    best_f=Y1.max(),
)
mt_candidate, _ = optimize_acqf(
    acq_function=mt_acquisition,
    bounds=bounds,
    q=1,
    num_restarts=2,
    raw_samples=16,
)
assert mt_candidate.shape == (1, d)
print("SAAS MultiTask candidate:", mt_candidate)
print("Task 1 objective:", 0.7 * f(mt_candidate) + 0.2)

# The categorical feature is stored in raw input space.
mixed_category = (torch.arange(n) % 2).to(dtype).unsqueeze(-1)
mixed_X = torch.cat([train_X, mixed_category], dim=-1)
mixed_Y = f(train_X) + 0.1 * mixed_category
mixed_model = MixedSaasFullyBayesianSingleTaskGP(
    mixed_X, mixed_Y, cat_dims=[d],
)
mixed_model.fit(
    warmup_steps=16,
    num_samples=8,
    thinning=2,
    disable_progbar=True,
)
mixed_bounds = torch.stack(
    [torch.zeros(d + 1, dtype=dtype), torch.ones(d + 1, dtype=dtype)]
)
mixed_acquisition = qLogExpectedImprovement(
    model=mixed_model, best_f=mixed_Y.max(),
)
mixed_candidate, _ = optimize_acqf_mixed(
    acq_function=mixed_acquisition,
    bounds=mixed_bounds,
    q=1,
    num_restarts=2,
    raw_samples=16,
    fixed_features_list=[{d: 0.0}, {d: 1.0}],
)
assert mixed_candidate[0, d].item() in (0.0, 1.0)
print("Mixed SAAS candidate:", mixed_candidate)
print("Mixed objective:", f(mixed_candidate[:, :d]) + 0.1 * mixed_candidate[:, d:])

# Mixed MultiTask: raw features are [design, category, task].
mixed_mt_X = torch.cat(
    [train_X_mt[:, :d], mixed_category[:len(x_mt)].repeat(2, 1), train_X_mt[:, d:]],
    dim=-1,
)
mixed_mt_Y = train_Y_mt + 0.1 * mixed_mt_X[:, d:d + 1]
mixed_mt_model = MixedSaasFullyBayesianMultiTaskGP(
    mixed_mt_X,
    mixed_mt_Y,
    task_feature=d + 1,
    cat_dims=[d],
    output_tasks=[1],
)
mixed_mt_model.fit(
    warmup_steps=16,
    num_samples=8,
    thinning=2,
    disable_progbar=True,
)
mixed_mt_acquisition = qLogExpectedImprovement(
    model=mixed_mt_model,
    best_f=mixed_mt_Y[len(x_mt):].max(),
)
mixed_mt_candidate, _ = optimize_acqf_mixed(
    acq_function=mixed_mt_acquisition,
    bounds=mixed_bounds,
    q=1,
    num_restarts=2,
    raw_samples=16,
    fixed_features_list=[{d: 0.0}, {d: 1.0}],
)
assert mixed_mt_candidate[0, d].item() in (0.0, 1.0)
mixed_mt_value = (
    0.7 * f(mixed_mt_candidate[:, :d])
    + 0.2
    + 0.1 * mixed_mt_candidate[:, d:]
)
print("Mixed MultiTask SAAS candidate:", mixed_mt_candidate)
print("Task 1 objective:", mixed_mt_value)


## 6. 追加実装例：提案候補を観測データへ追加する

実際のBOでは、候補を評価したら `train_X` と `train_Y` に観測を追加し、モデルを再学習します。ここでは**MCMCの追加実行時間を抑えるため、データ更新まで**を示します。更新済みのデータを使って新しいSAASモデルを構築し、必要に応じて `fit()` を実行してください。

新規モデルの構築だけでは事後分布は更新されません。学習済みモデルへデータを追加したことにもなりません。

In [ ]:
updated_X = torch.cat([train_X, candidate.detach()], dim=0)
updated_Y = torch.cat([train_Y, candidate_Y.detach()], dim=0)

assert updated_X.shape == (n + 1, d)
assert updated_Y.shape == (n + 1, 1)
print("updated SAAS training data:", updated_X.shape, updated_Y.shape)

# To continue BO, construct a new model from updated_X / updated_Y
# and call fit() with appropriate MCMC settings and convergence checks.